# Methods & Validation Notebook

This notebook runs a compact set of validation tests and plots comparing MoonStone solvers to analytic expectations. It is intended to be CI-friendly and illustrate how to pack/unpack `.moonstone.bundles` simulation files for reproducible verification.

In [ ]:
# Imports and helpers
import numpy as np
import matplotlib.pyplot as plt
from app import geodesics
from backend.app.bundle_utils import load_bundle, save_bundle, list_bundles

plt.rcParams['figure.figsize'] = (6,4)

def plot_trace_xy(pts, label=None):
    pts = np.array(pts)
    plt.plot(pts[:,0], pts[:,1], marker='.', markersize=3, label=label)
    plt.gca().set_aspect('equal','box')
    if label:
        plt.legend()


## Flat-space sanity test
The solver should reproduce straight-line propagation in flat spacetime (trace_flat). We check max deviation from ideal straight-line points.

In [ ]:
# Run flat trace and compare to analytic straight-line
src = (0.0, 0.0, 0.0)
dir = (1.0, 0.0, 0.0)
pts = geodesics.trace_flat(src, dir, {'npoints': 200, 'step': 0.01})
# analytic straight-line
t = np.arange(200) * 0.01
x_exp = src[0] + dir[0] * t
y_exp = src[1] + dir[1] * t
pts_arr = np.array(pts)
max_dev = np.max(np.sqrt((pts_arr[:,0] - x_exp)**2 + (pts_arr[:,1] - y_exp)**2))
print('Flat-space max deviation:', max_dev)
plt.figure()
plot_trace_xy(pts, label='computed')
plt.plot(x_exp, y_exp, '-', label='expected', alpha=0.6)
plt.title('Flat-space sanity test')
plt.show()

## Schwarzschild small-deflection test
Compare the weak-field trace deflection to the analytical small-angle formula \(\Delta\phi \approx 4M/b\). We'll measure asymptotic incoming/outgoing angles and compute the difference.

In [ ]:
# Weak-field deflection example
M = 1e-3
b = 10.0  # impact parameter (large -> weak field)
src = (-200.0, b, 0.0)
dir = (1.0, 0.0, 0.0)
pts = geodesics.trace_schwarzschild_weak(src, dir, mass=M, params={'npoints': 1200, 'step': 0.2})
pts = np.array(pts)
# compute incoming slope from first chunk and outgoing slope from last chunk
def asymptotic_angle(seg):
    xs = seg[:,0]; ys = seg[:,1]
    coeff = np.polyfit(xs, ys, 1)
    return np.arctan(coeff[0])

angle_in = asymptotic_angle(pts[:50])
angle_out = asymptotic_angle(pts[-50:])
delta = abs(angle_out - angle_in)
analytic = 4.0 * M / b
print(f"Measured deflection (rad): {delta:.6e}, Analytic approx 4M/b: {analytic:.6e}, relative: {abs(delta-analytic)/analytic:.2%}")
plt.figure()
plot_trace_xy(pts, label='trajectory')
plt.title('Schwarzschild weak-field deflection')
plt.show()

## Nullness check
Check that null geodesics maintain an approximate null condition under the weak-field approximation (i.e., spatial propagation consistent with ut estimate).

In [ ]:
# Run null_formal and check approximate null condition
M = 1e-3
b = 5.0
src = (-50.0, b, 0.0)
dir = (1.0, 0.0, 0.0)
pts = geodesics.trace_null_formal(src, dir, mass=M, params={'npoints': 800, 'step': 0.05})
pts = np.array(pts)
# approximate spatial velocity (finite differences) per step
dpos = np.sqrt(np.sum(np.diff(pts, axis=0)**2, axis=1))
# approximate "spatial speed" per lambda step (assuming step ~ lambda)
v = np.concatenate([[dpos[0]], dpos]) / 0.05
# approximate local Phi and ut
rs = np.sqrt(pts[:,0]**2 + pts[:,1]**2 + pts[:,2]**2) + 1e-12
Phi = -M / rs
ut_est = 1.0 / np.sqrt(np.maximum(1e-12, 1.0 - 2.0*Phi))
# check ratio v / ut_est ~ 1
ratios = v / ut_est
print('Nullness check: median ratio v/ut:', np.median(ratios), 'max dev from 1:', np.max(np.abs(ratios-1.0)))
plt.figure()
plot_trace_xy(pts, label='null trajectory')
plt.title('Nullness check (weak-field)')
plt.show()

## Bundles: load default example and run
Load the sample bundle from `.moonstone.bundles/default_simulation.json` and run the described simulation using CPU fallbacks when needed.

In [ ]:
# Load a sample bundle and run
bundle = load_bundle('default_simulation')
print('Bundle:', bundle.get('name'), '-', bundle.get('description'))
# choose method-based runner
method = bundle['params'].get('method')
if method == 'kerr_formal':
    # run CPU fallback
    pts = geodesics.trace_kerr_formal(tuple(bundle['source'].values()), tuple(bundle['directions'][0].values()), bundle['metric'].get('mass', 1.0), tuple(bundle['params'].get('spin',(0.0,0.0,0.0))), bundle['params'])
else:
    pts = geodesics.trace_schwarzschild_batch(tuple(bundle['source'].values()), [tuple(d.values()) for d in bundle['directions']], bundle['metric'].get('mass',1.0), bundle['params'])[0]
plt.figure()
plot_trace_xy(pts, label='bundle run')
plt.title(f"Bundle run: {bundle.get('name')}")
plt.show()

In [ ]:
# Save a small output artifact (example)
save_bundle('last_bundle_run_result', {'name': 'last_run', 'points': pts[:10].tolist()})
print('Saved run artifact to .moonstone.bundles/last_bundle_run_result.json')
print('Available bundles:', list_bundles())